# Visual Product Search Engine — Evaluation (Config C)

This notebook evaluates retrieval performance for Config C (fine-tuned CLIP).

---

### Metrics computed
| Metric | Meaning |
|--------|---------|
| Recall@K | Fraction of queries with at least 1 correct result in top-K |
| NDCG@K | Position-aware metric — correct results ranked higher = better score |
| mAP@K | Mean average precision up to K — rewards finding all correct items early |

K values: 5, 10, 15

---

### How evaluation works
1. For each query image, encode it with CLIP (image only — no caption for query)
2. Search the gallery FAISS index → top-K results
3. Check which results share the same item_id as the query → ground truth match
4. Compute metrics over all queries
5. Report mean ± std over multiple seeds (your roll numbers)

---

### Inputs
- `yolo-bbox-crops-v1` — query crops + master_crops.csv
- `clip-finetuned-indexes-c` — Config C FAISS indexes + gallery_metadata.csv


## Step 1: Install and Import

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install ftfy regex transformers faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.7 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print('Libraries imported!')

Device : cuda
Libraries imported!


## Step 2: Paths and Seeds

**IMPORTANT: Set SEEDS to your team's roll numbers before running.**

In [3]:
# ============================================================
#  SET YOUR TEAM ROLL NUMBERS AS SEEDS
# ============================================================
SEEDS = [550, 537, 585, 35]   # team roll numbers
# ============================================================

CROPS_DATASET        = '/kaggle/input/datasets/harshitabansal307/yolo-bbox-crops-v1'
GALLERY_META_DATASET = '/kaggle/input/datasets/likithareddy2508/clip-indexes-ab'

# Print ALL available input paths so we can find the exact dataset name
print("=== All available input paths ===")
for root in ['/kaggle/input', '/kaggle/input/datasets']:
    if not os.path.exists(root):
        continue
    for entry in os.listdir(root):
        full = os.path.join(root, entry)
        print(f"  {full}")
        if os.path.isdir(full):
            try:
                for sub in os.listdir(full):
                    print(f"    {os.path.join(full, sub)}")
                    sub_full = os.path.join(full, sub)
                    if os.path.isdir(sub_full):
                        for subsub in os.listdir(sub_full):
                            print(f"      {os.path.join(sub_full, subsub)}")
            except:
                pass
print("=================================")

# Auto-detect: find any folder containing the finetuned index files
def find_dataset_by_file(filename):
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return root
    return None

INDEXES_DATASET = find_dataset_by_file('faiss_index_C_alpha07.bin')
if INDEXES_DATASET is None:
    raise FileNotFoundError(
        "Could not find faiss_index_C_alpha07.bin anywhere in /kaggle/input. "
        "Make sure clip-finetuned-indexes-c is added as an input dataset."
    )

K_VALUES   = [5, 10, 15]
BATCH_SIZE = 64
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'

CONFIGS = [
    ('Config_C_alpha0.7',  'faiss_index_C_alpha07.bin', 0.7),
    ('Config_C_alpha0.5',  'faiss_index_C_alpha05.bin', 0.5),
]

for label, path in [('CROPS_DATASET', CROPS_DATASET), ('INDEXES_DATASET', INDEXES_DATASET), ('GALLERY_META_DATASET', GALLERY_META_DATASET)]:
    status = 'Found ✓' if os.path.exists(path) else 'NOT FOUND ✗'
    print(f'[{status}] {label}: {path}')

print(f'\nSeeds : {SEEDS}')
print(f'K values : {K_VALUES}')


=== All available input paths ===
  /kaggle/input/datasets
    /kaggle/input/datasets/likithareddy2508
      /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c
      /kaggle/input/datasets/likithareddy2508/clip-indexes-ab
    /kaggle/input/datasets/harshitabansal307
      /kaggle/input/datasets/harshitabansal307/yolo-bbox-crops-v1
      /kaggle/input/datasets/harshitabansal307/blipcaptionsoutput
  /kaggle/input/datasets/likithareddy2508
    /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c
      /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c/faiss_index_C_alpha07.bin
      /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c/clip_finetuned_vision.pt
      /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c/faiss_index_C_alpha05.bin
      /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c/training_curve.png
      /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c/clip_finetuned_full.pt
  

## Step 3: Load Query Data and Gallery Metadata

In [4]:
master_df = pd.read_csv(os.path.join(CROPS_DATASET, 'master_crops.csv'))
query_df  = master_df[master_df['split'] == 'query'].reset_index(drop=True)

# Remap query crop paths
def remap_path(old_path):
    if pd.isna(old_path):
        return old_path
    relative = old_path.replace('/kaggle/working/', '').replace('/kaggle/input/', '')
    for ds in ['yolocroppeddataset/', 'yolo-bbox-crops-v1/', 'datasets/harshitabansal307/yolo-bbox-crops-v1/']:
        relative = relative.replace(ds, '')
    return os.path.join(CROPS_DATASET, relative)

query_df['crop_path_new'] = query_df['crop_path'].apply(remap_path)
query_df['crop_exists']   = query_df['crop_path_new'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

if query_df['crop_exists'].sum() < len(query_df) * 0.9:
    def direct_path(image_name):
        relative = image_name[4:] if image_name.startswith('img/') else image_name
        for subdir in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(CROPS_DATASET, subdir, relative)
            if os.path.exists(p):
                return p
        return os.path.join(CROPS_DATASET, 'data/bbox_crops', relative)
    query_df['crop_path_new'] = query_df['image_name'].apply(direct_path)
    query_df['crop_exists']   = query_df['crop_path_new'].apply(os.path.exists)

valid_query_df = query_df[query_df['crop_exists']].reset_index(drop=True)

# Load gallery metadata
gallery_meta = pd.read_csv(os.path.join(GALLERY_META_DATASET, 'gallery_metadata.csv'))

print(f'Query images  : {len(valid_query_df):,}')
print(f'Gallery items : {len(gallery_meta):,}')
print(f'Unique query item_ids : {valid_query_df["item_id"].nunique():,}')

Query images  : 14,218
Gallery items : 12,612
Unique query item_ids : 3,985


## Step 4: Load CLIP and Encode All Query Images

In [5]:
print('Loading CLIP...')
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)

# Load fine-tuned weights
ft_weights_path = os.path.join(INDEXES_DATASET, 'clip_finetuned_full.pt')
print(f'Loading fine-tuned weights from: {ft_weights_path}')
state_dict = torch.load(ft_weights_path, map_location=DEVICE)
clip_model.load_state_dict(state_dict)
print('Fine-tuned weights loaded ✓')

for param in clip_model.parameters():
    param.requires_grad = False
clip_model.eval()

EMBED_DIM = clip_model.config.projection_dim
print(f'CLIP loaded! Embedding dim: {EMBED_DIM}')

def get_image_emb(inputs):
    """
    Correctly extract projected, normalised image embeddings from CLIP.
    In newer transformers, get_image_features() returns BaseModelOutputWithPooling,
    not a plain tensor. We go through vision_model + visual_projection explicitly.
    """
    vision_out = clip_model.vision_model(pixel_values=inputs['pixel_values'])
    pooled = vision_out.pooler_output          # (B, hidden_size)
    projected = clip_model.visual_projection(pooled)  # (B, EMBED_DIM)
    projected = projected / projected.norm(dim=-1, keepdim=True)  # L2 normalise
    return projected  # plain tensor, always

def encode_images(image_list):
    inputs = clip_processor(images=image_list, return_tensors='pt', padding=True).to(DEVICE)
    with torch.no_grad():
        feats = get_image_emb(inputs)
    return feats.cpu().numpy()

def encode_texts(text_list):
    inputs = clip_processor(
        text=text_list, return_tensors='pt', padding=True, truncation=True, max_length=77
    ).to(DEVICE)
    with torch.no_grad():
        text_out = clip_model.text_model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        pooled = text_out.pooler_output
        projected = clip_model.text_projection(pooled)
        projected = projected / projected.norm(dim=-1, keepdim=True)
    return projected.cpu().numpy()

# Sanity check
import PIL
dummy_emb = encode_images([PIL.Image.new('RGB', (224, 224))])
norm = float(np.linalg.norm(dummy_emb[0]))
print(f'Embedding norm check: {norm:.4f}  (must be ~1.0)')
assert 0.99 < norm < 1.01, f"Norm is {norm:.4f} — something is wrong!"
print('Norm check passed ✓')

# Encode all query images
n_query = len(valid_query_df)
query_image_embs = np.zeros((n_query, EMBED_DIM), dtype=np.float32)

print(f'Encoding {n_query:,} query images with fine-tuned CLIP...')
for batch_start in tqdm(range(0, n_query, BATCH_SIZE), desc='Query encoding'):
    batch = valid_query_df.iloc[batch_start : batch_start + BATCH_SIZE]
    images, valid_idx = [], []
    for i, (_, row) in enumerate(batch.iterrows()):
        try:
            images.append(Image.open(row['crop_path_new']).convert('RGB'))
            valid_idx.append(i)
        except:
            pass
    if not images:
        continue
    embs = encode_images(images)
    for local_i, global_i in enumerate(valid_idx):
        query_image_embs[batch_start + global_i] = embs[local_i]

print(f'Query embeddings shape : {query_image_embs.shape}')
print(f'Sample norm            : {np.linalg.norm(query_image_embs[0]):.4f}  (should be ~1.0)')


Loading CLIP...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading fine-tuned weights from: /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c/clip_finetuned_full.pt
Fine-tuned weights loaded ✓
CLIP loaded! Embedding dim: 512
Embedding norm check: 1.0000  (must be ~1.0)
Norm check passed ✓
Encoding 14,218 query images with fine-tuned CLIP...


Query encoding: 100%|██████████| 223/223 [03:51<00:00,  1.04s/it]

Query embeddings shape : (14218, 512)
Sample norm            : 1.0000  (should be ~1.0)


In [6]:
# ── Rebuild Config C gallery indexes with corrected encoding ────────────────
import faiss

print('Loading gallery data for re-encoding...')
gallery_df = master_df[master_df['split'] == 'gallery'].reset_index(drop=True)

def remap_path(old_path):
    if pd.isna(old_path):
        return old_path
    relative = old_path.replace('/kaggle/working/', '').replace('/kaggle/input/', '')
    for ds in ['yolocroppeddataset/', 'yolo-bbox-crops-v1/', 'datasets/harshitabansal307/yolo-bbox-crops-v1/']:
        relative = relative.replace(ds, '')
    return os.path.join(CROPS_DATASET, relative)

gallery_df['crop_path_new'] = gallery_df['crop_path'].apply(remap_path)
gallery_df['crop_exists']   = gallery_df['crop_path_new'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)
valid_gallery = gallery_df[gallery_df['crop_exists']].reset_index(drop=True)
print(f'Gallery images found: {len(valid_gallery):,}')

CAPTIONS_DATASET = '/kaggle/input/datasets/harshitabansal307/blipcaptionsoutput'
with open(os.path.join(CAPTIONS_DATASET, 'gallery_captions.json')) as f:
    gallery_captions = json.load(f)

BATCH_SIZE_EMB = 64
n_gallery = len(valid_gallery)
ft_image_embs = np.zeros((n_gallery, EMBED_DIM), dtype=np.float32)
ft_text_embs  = np.zeros((n_gallery, EMBED_DIM), dtype=np.float32)

print(f'Re-encoding {n_gallery:,} gallery images + captions...')
for batch_start in tqdm(range(0, n_gallery, BATCH_SIZE_EMB), desc='Gallery re-encoding'):
    batch = valid_gallery.iloc[batch_start : batch_start + BATCH_SIZE_EMB]

    images, valid_idx = [], []
    for i, (_, row) in enumerate(batch.iterrows()):
        try:
            images.append(Image.open(row['crop_path_new']).convert('RGB'))
            valid_idx.append(i)
        except:
            pass
    if images:
        embs = encode_images(images)
        for local_i, global_i in enumerate(valid_idx):
            ft_image_embs[batch_start + global_i] = embs[local_i]

    texts = [gallery_captions.get(row['image_name'], 'a clothing item') for _, row in batch.iterrows()]
    t_embs = encode_texts(texts)
    ft_text_embs[batch_start : batch_start + len(texts)] = t_embs

print('Re-encoding done!')
print(f'Image emb norm: {np.linalg.norm(ft_image_embs[0]):.4f}  (should be ~1.0)')
print(f'Text  emb norm: {np.linalg.norm(ft_text_embs[0]):.4f}  (should be ~1.0)')

OUTPUT_DIR = '/kaggle/working'
for alpha in [0.7, 0.5]:
    fused = alpha * ft_image_embs + (1 - alpha) * ft_text_embs
    norms = np.linalg.norm(fused, axis=-1, keepdims=True)
    fused = fused / np.maximum(norms, 1e-8)
    alpha_str = f'alpha{int(alpha*10):02d}'
    np.save(f'{OUTPUT_DIR}/gallery_embeddings_C_{alpha_str}.npy', fused)
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(fused.astype(np.float32))
    faiss.write_index(index, f'{OUTPUT_DIR}/faiss_index_C_{alpha_str}.bin')
    print(f'Config C index rebuilt: alpha={alpha}, vectors={index.ntotal:,}')

INDEXES_DATASET_REBUILT = OUTPUT_DIR
print(f'Rebuilt indexes saved to {OUTPUT_DIR}')


Loading gallery data for re-encoding...
Gallery images found: 12,612
Re-encoding 12,612 gallery images + captions...


Gallery re-encoding: 100%|██████████| 198/198 [02:34<00:00,  1.28it/s]


Re-encoding done!
Image emb norm: 1.0000  (should be ~1.0)
Text  emb norm: 1.0000  (should be ~1.0)
Config C index rebuilt: alpha=0.7, vectors=12,612
Config C index rebuilt: alpha=0.5, vectors=12,612
Rebuilt indexes saved to /kaggle/working


## Step 5: Define Metric Functions

In [7]:
# Metric functions — kept for reference, but evaluation loop uses inline logic
def recall_at_k(retrieved_ids, relevant_id, k):
    return int(any(r == relevant_id for r in retrieved_ids[:k]))

print('Metric functions defined ✓')
test_retrieved = ['A', 'B', 'C', 'D', 'E']
print(f'Test Recall@5: {recall_at_k(test_retrieved, "A", 5)}  (expected 1)')
print(f'Test Recall@5: {recall_at_k(test_retrieved, "Z", 5)}  (expected 0)')


Metric functions defined ✓
Test Recall@5: 1  (expected 1)
Test Recall@5: 0  (expected 0)


## Step 6: Run Evaluation for Each Config and Each Seed

In [8]:
gallery_item_ids = gallery_meta['item_id'].tolist()

# Pre-build a lookup: item_id -> count of gallery images with that item_id
gallery_item_counts = gallery_meta['item_id'].value_counts().to_dict()

all_results = []

for config_name, index_file, alpha in CONFIGS:
    print(f'\n=== Evaluating {config_name} ===')

    index_path = os.path.join('/kaggle/working', index_file)
    index = faiss.read_index(index_path)
    print(f'  Index loaded: {index.ntotal:,} vectors')

    search_embs = query_image_embs.astype(np.float32)
    seed_scores = {k: {'recall': [], 'ndcg': [], 'map': []} for k in K_VALUES}

    for seed in SEEDS:
        np.random.seed(seed)
        torch.manual_seed(seed)

        n_sample = max(500, int(0.2 * len(valid_query_df)))
        n_sample = min(n_sample, len(valid_query_df))
        sample_indices = np.random.choice(len(valid_query_df), n_sample, replace=False)

        sample_embs     = search_embs[sample_indices]
        sample_item_ids = valid_query_df.iloc[sample_indices]['item_id'].tolist()

        distances, indices = index.search(sample_embs, 15 + 1)

        per_k_recall = {k: [] for k in K_VALUES}
        per_k_ndcg   = {k: [] for k in K_VALUES}
        per_k_map    = {k: [] for k in K_VALUES}

        for q_item_id, result_indices in zip(sample_item_ids, indices):
            retrieved_item_ids = [gallery_item_ids[i] for i in result_indices if i < len(gallery_item_ids)]

            # relevant_ids: the set of item_ids that match (just the query's item_id)
            relevant_ids = {q_item_id}

            # n_relevant: how many gallery images actually have this item_id
            # This is the correct denominator for NDCG and mAP
            n_relevant_in_gallery = gallery_item_counts.get(q_item_id, 1)

            for k in K_VALUES:
                retrieved_k = retrieved_item_ids[:k]

                # Recall@K: 1 if any match in top-k
                rec = int(any(r == q_item_id for r in retrieved_k))
                per_k_recall[k].append(rec)

                # NDCG@K: use n_relevant_in_gallery as the ideal count
                dcg = sum(
                    1.0 / np.log2(rank + 2)
                    for rank, r in enumerate(retrieved_k)
                    if r == q_item_id
                )
                n_ideal = min(n_relevant_in_gallery, k)
                idcg = sum(1.0 / np.log2(i + 2) for i in range(n_ideal))
                per_k_ndcg[k].append(dcg / idcg if idcg > 0 else 0.0)

                # mAP@K: divide by min(n_relevant_in_gallery, k)
                hits, prec_sum = 0, 0.0
                for rank, r in enumerate(retrieved_k, start=1):
                    if r == q_item_id:
                        hits += 1
                        prec_sum += hits / rank
                denom = min(n_relevant_in_gallery, k)
                per_k_map[k].append(prec_sum / denom if denom > 0 else 0.0)

        for k in K_VALUES:
            seed_scores[k]['recall'].append(np.mean(per_k_recall[k]))
            seed_scores[k]['ndcg'].append(np.mean(per_k_ndcg[k]))
            seed_scores[k]['map'].append(np.mean(per_k_map[k]))

        print(f'  Seed {seed}: Recall@10={np.mean(per_k_recall[10]):.4f}  NDCG@10={np.mean(per_k_ndcg[10]):.4f}  mAP@10={np.mean(per_k_map[10]):.4f}')

    for k in K_VALUES:
        all_results.append({
            'config': config_name, 'K': k,
            'Recall_mean': np.mean(seed_scores[k]['recall']),
            'Recall_std':  np.std(seed_scores[k]['recall']),
            'NDCG_mean':   np.mean(seed_scores[k]['ndcg']),
            'NDCG_std':    np.std(seed_scores[k]['ndcg']),
            'mAP_mean':    np.mean(seed_scores[k]['map']),
            'mAP_std':     np.std(seed_scores[k]['map']),
        })

print('\nEvaluation complete!')



=== Evaluating Config_C_alpha0.7 ===
  Index loaded: 12,612 vectors
  Seed 550: Recall@10=0.9191  NDCG@10=0.6511  mAP@10=0.5586
  Seed 537: Recall@10=0.9142  NDCG@10=0.6441  mAP@10=0.5499
  Seed 585: Recall@10=0.9131  NDCG@10=0.6485  mAP@10=0.5559
  Seed 35: Recall@10=0.9107  NDCG@10=0.6447  mAP@10=0.5516

=== Evaluating Config_C_alpha0.5 ===
  Index loaded: 12,612 vectors
  Seed 550: Recall@10=0.9187  NDCG@10=0.6438  mAP@10=0.5501
  Seed 537: Recall@10=0.9163  NDCG@10=0.6369  mAP@10=0.5405
  Seed 585: Recall@10=0.9131  NDCG@10=0.6412  mAP@10=0.5467
  Seed 35: Recall@10=0.9093  NDCG@10=0.6375  mAP@10=0.5443

Evaluation complete!


## Step 7: Print Results Table

In [9]:
results_df = pd.DataFrame(all_results)

print('\n=== CONFIG C EVALUATION RESULTS ===')
print(f'Seeds used: {SEEDS}')
print(f'Format: mean ± std')
print()

for config_name, _, _ in CONFIGS:
    config_rows = results_df[results_df['config'] == config_name]
    print(f'Config: {config_name}')
    print(f'  {"K":>4}  {"Recall@K":>12}  {"NDCG@K":>12}  {"mAP@K":>12}')
    print(f'  {"─"*50}')
    for _, row in config_rows.iterrows():
        print(
            f'  K={int(row["K"]):>2}  '
            f'{row["Recall_mean"]:.4f}±{row["Recall_std"]:.4f}  '
            f'{row["NDCG_mean"]:.4f}±{row["NDCG_std"]:.4f}  '
            f'{row["mAP_mean"]:.4f}±{row["mAP_std"]:.4f}'
        )
    print()

# Save results
results_df.to_csv('/kaggle/working/evaluation_results_C.csv', index=False)
print('Results saved to evaluation_results_C.csv')


=== CONFIG C EVALUATION RESULTS ===
Seeds used: [550, 537, 585, 35]
Format: mean ± std

Config: Config_C_alpha0.7
     K      Recall@K        NDCG@K         mAP@K
  ──────────────────────────────────────────────────
  K= 5  0.8718±0.0035  0.6326±0.0033  0.5577±0.0036
  K=10  0.9143±0.0031  0.6471±0.0028  0.5540±0.0035
  K=15  0.9357±0.0025  0.6612±0.0030  0.5594±0.0035

Config: Config_C_alpha0.5
     K      Recall@K        NDCG@K         mAP@K
  ──────────────────────────────────────────────────
  K= 5  0.8709±0.0042  0.6241±0.0027  0.5478±0.0029
  K=10  0.9144±0.0036  0.6399±0.0028  0.5454±0.0035
  K=15  0.9334±0.0022  0.6550±0.0026  0.5515±0.0035

Results saved to evaluation_results_C.csv
